# RAG 데이터 임베딩 및 테스트 노트북

문서를 읽고 분할한 뒤 Pinecone에 벡터로 저장하고 검색을 테스트합니다.

In [2]:
%pip install python-docx langchain langchain-community langchain-huggingface langchain-pinecone

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

load_dotenv()

C:\Users\darkk\AppData\Local\Temp\ipykernel_18284\96430379.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader
c:\Users\darkk\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

### 1. 워드 문서 로드

In [ ]:
file_path = "./tax_with_table.docx"
loader = Docx2txtLoader(file_path)
docs = loader.load()
print(f"문서 로드 완료: {len(docs)} 개의 문서")

### 2. 텍스트 분할 (Chunking)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splits = text_splitter.split_documents(docs)
print(f"텍스트 분할 완료: 총 {len(splits)} 개의 청크")

### 3. 임베딩 모델 로드 및 Pinecone 업로드
*주의: intfloat/multilingual-e5-large 모델 다운로드에 시간이 걸릴 수 있습니다.*

In [1]:
embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")

index_name = "tax-chatbot"
vectorstore = PineconeVectorStore.from_documents(
    splits,
    embeddings,
    index_name=index_name
)
print("Pinecone 업로드 및 인덱스 연결 완료")

NameError: name 'HuggingFaceEmbeddings' is not defined

### 4. 검색(Retrieval) 테스트

In [ ]:
retriever = vectorstore.as_retriever()
query = "연봉 5000만원의 소득세는 어떻게 계산하나요?"

results = retriever.invoke(query)
print(f"검색된 관련 문서 수: {len(results)}\n")

for i, res in enumerate(results):
    print(f"--- [문서 {i+1}] ---")
    print(res.page_content[:300] + "...")
    print("\n")